# AWS DMS: PostgreSQL RDS → S3

Notebook này thực hiện **one-time full load**: sao chép các dòng cũ từ một bảng PostgreSQL sang S3 ở định dạng Parquet.

Bạn chỉ cần dùng 3 hàm:

1. `setup()` — tạo/tái sử dụng hạ tầng, kiểm tra kết nối và tự bắt đầu sao chép.
2. `status()` — xem tiến độ, lỗi của bảng và các file đầu tiên trên S3.
3. `destroy()` — xóa tài nguyên DMS để ngừng phát sinh chi phí.

> `destroy()` **không xóa RDS, S3 bucket hoặc dữ liệu đã ghi lên S3**.

Cách chạy an toàn: **Restart Kernel → Run All → gọi `setup()` ở cell cuối**. Các cell định nghĩa hàm không tự tạo hay xóa tài nguyên AWS.


## 1. Cấu hình

Tạo file `dms/.env` từ `dms/.env.example`. Các biến quan trọng:

- `SUBNET_IDS`: ít nhất 2 subnet trong cùng VPC với RDS.
- `SECURITY_GROUP_IDS`: security group của DMS; RDS phải cho phép group này truy cập port PostgreSQL.
- `DATE_COLUMN`: cột dùng để xác định dữ liệu cũ, ví dụ `created_at_utc` hoặc `closed_at_utc`.
- `ARCHIVE_RETENTION_DAYS=90`: chỉ lấy các dòng có `DATE_COLUMN` cũ hơn 90 ngày.

Nếu mục tiêu là chỉ archive đơn hàng đã hoàn tất, nên dùng `closed_at_utc`. Dùng `created_at_utc` sẽ lấy mọi đơn hàng cũ, kể cả đơn chưa đóng nếu bảng còn loại dữ liệu đó.

Notebook đọc `.env` mỗi lần gọi một trong 3 hàm, vì vậy sửa `.env` không cần restart kernel.


In [1]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import boto3
from botocore.exceptions import ClientError

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None


@dataclass(frozen=True)
class Config:
    """Toàn bộ cấu hình cần cho pipeline."""

    aws_region: str
    rds_id: str
    rds_host: str
    rds_port: int
    rds_database: str
    rds_username: str
    rds_password: str
    subnet_ids: list[str]
    security_group_ids: list[str]
    source_schema: str
    source_table: str
    date_column: str
    retention_days: int
    s3_bucket: str
    s3_prefix: str
    dms_prefix: str
    dms_instance_class: str
    dms_storage_gb: int

    @property
    def instance_id(self) -> str:
        return f"{self.dms_prefix}-instance"

    @property
    def subnet_group_id(self) -> str:
        return f"{self.dms_prefix}-subnet"

    @property
    def source_endpoint_id(self) -> str:
        return f"{self.dms_prefix}-source"

    @property
    def target_endpoint_id(self) -> str:
        return f"{self.dms_prefix}-target"

    @property
    def task_id(self) -> str:
        return f"{self.dms_prefix}-task"

    @property
    def s3_role_name(self) -> str:
        return f"{self.dms_prefix}-s3-role"


@dataclass(frozen=True)
class Aws:
    """Nhóm AWS clients để truyền giữa các helper."""

    s3: Any
    dms: Any
    iam: Any
    ec2: Any


def _read_env() -> None:
    """Tìm .env khi Jupyter chạy từ project root hoặc thư mục dms/."""
    candidates = [Path.cwd() / ".env", Path.cwd() / "dms" / ".env"]
    env_path = next((path for path in candidates if path.is_file()), None)
    if env_path is None:
        return  # Vẫn cho phép dùng environment variables của hệ điều hành.
    if load_dotenv is None:
        raise ImportError("Thiếu python-dotenv. Cài bằng: pip install python-dotenv")
    load_dotenv(env_path, override=True)


def _required(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"Thiếu biến môi trường {name} trong dms/.env")
    return value


def _positive_int(name: str, default: str) -> int:
    raw_value = os.getenv(name, default).strip()
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} phải là số nguyên, nhận được {raw_value!r}") from error
    if value <= 0:
        raise ValueError(f"{name} phải lớn hơn 0")
    return value


def _identifier(name: str, value: str) -> str:
    """Chặn nhầm tên schema/table/column trước khi gửi table mapping tới DMS."""
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", value):
        raise ValueError(f"{name} không phải PostgreSQL identifier hợp lệ: {value!r}")
    return value


def _context() -> tuple[Config, Aws]:
    """Đọc cấu hình mới nhất và tạo AWS clients."""
    _read_env()
    config = Config(
        aws_region=_required("AWS_REGION"),
        rds_id=_required("RDS_ID"),
        rds_host=_required("RDS_HOST"),
        rds_port=_positive_int("RDS_PORT", "5432"),
        rds_database=_required("RDS_DATABASE"),
        rds_username=_required("RDS_USERNAME"),
        rds_password=_required("RDS_PASSWORD"),
        subnet_ids=[item.strip() for item in _required("SUBNET_IDS").split(",") if item.strip()],
        security_group_ids=[item.strip() for item in _required("SECURITY_GROUP_IDS").split(",") if item.strip()],
        source_schema=_identifier("SOURCE_SCHEMA", os.getenv("SOURCE_SCHEMA", "public").strip()),
        source_table=_identifier("SOURCE_TABLE", os.getenv("SOURCE_TABLE", "orders").strip()),
        date_column=_identifier("DATE_COLUMN", os.getenv("DATE_COLUMN", "closed_at_utc").strip()),
        retention_days=_positive_int("ARCHIVE_RETENTION_DAYS", "90"),
        s3_bucket=_required("S3_BUCKET"),
        s3_prefix=_required("S3_PREFIX").strip("/"),
        dms_prefix=_required("DMS_PREFIX"),
        dms_instance_class=os.getenv("DMS_INSTANCE_CLASS", "dms.t3.medium").strip(),
        dms_storage_gb=_positive_int("DMS_STORAGE_GB", "50"),
    )
    if len(config.subnet_ids) < 2:
        raise ValueError("SUBNET_IDS cần ít nhất 2 subnet cho DMS subnet group")
    if not config.security_group_ids:
        raise ValueError("SECURITY_GROUP_IDS không được rỗng")

    session = boto3.Session(region_name=config.aws_region)
    aws = Aws(
        s3=session.client("s3"),
        dms=session.client("dms"),
        iam=session.client("iam"),
        ec2=session.client("ec2"),
    )
    return config, aws


## 2. Helper nội bộ

Các hàm bắt đầu bằng `_` là chi tiết triển khai. Người dùng notebook không cần gọi trực tiếp; `setup()`, `status()` và `destroy()` sẽ gọi chúng đúng thứ tự.


In [2]:
from __future__ import annotations

DMS_VPC_ROLE_NAME = "dms-vpc-role"
DMS_VPC_POLICY_ARN = "arn:aws:iam::aws:policy/service-role/AmazonDMSVPCManagementRole"


def _find_instance(config: Config, aws: Aws) -> dict[str, Any] | None:
    try:
        items = aws.dms.describe_replication_instances(
            Filters=[{"Name": "replication-instance-id", "Values": [config.instance_id]}]
        )["ReplicationInstances"]
    except aws.dms.exceptions.ResourceNotFoundFault:
        return None  # Resource vừa bị xóa trong lúc destroy() poll.
    return items[0] if items else None


def _find_endpoint(endpoint_id: str, aws: Aws) -> dict[str, Any] | None:
    try:
        items = aws.dms.describe_endpoints(
            Filters=[{"Name": "endpoint-id", "Values": [endpoint_id]}]
        )["Endpoints"]
    except aws.dms.exceptions.ResourceNotFoundFault:
        return None
    return items[0] if items else None


def _find_task(config: Config, aws: Aws) -> dict[str, Any] | None:
    try:
        items = aws.dms.describe_replication_tasks(
            Filters=[{"Name": "replication-task-id", "Values": [config.task_id]}]
        )["ReplicationTasks"]
    except aws.dms.exceptions.ResourceNotFoundFault:
        return None
    return items[0] if items else None


def _wait_for(
    description: str,
    fetch: Any,
    is_done: Any,
    timeout_seconds: int = 900,
    interval_seconds: int = 10,
) -> Any:
    """Poll AWS với timeout để notebook không chờ vô hạn."""
    deadline = time.monotonic() + timeout_seconds
    while True:
        value = fetch()
        if is_done(value):
            return value
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Hết thời gian chờ: {description}")
        print(f"  Đang chờ {description}...")
        time.sleep(interval_seconds)


def _ensure_bucket(config: Config, aws: Aws) -> None:
    try:
        aws.s3.head_bucket(Bucket=config.s3_bucket)
        print(f"✓ S3 bucket đã tồn tại: {config.s3_bucket}")
    except ClientError as error:
        code = error.response.get("Error", {}).get("Code", "")
        if code not in {"404", "NoSuchBucket", "NotFound"}:
            raise
        arguments: dict[str, Any] = {"Bucket": config.s3_bucket}
        if config.aws_region != "us-east-1":
            arguments["CreateBucketConfiguration"] = {"LocationConstraint": config.aws_region}
        aws.s3.create_bucket(**arguments)
        print(f"✓ Đã tạo S3 bucket: {config.s3_bucket}")

    aws.s3.put_public_access_block(
        Bucket=config.s3_bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True,
            "IgnorePublicAcls": True,
            "BlockPublicPolicy": True,
            "RestrictPublicBuckets": True,
        },
    )


def _ensure_iam_roles(config: Config, aws: Aws) -> str:
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "dms.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }],
    }

    created_role = False
    try:
        aws.iam.get_role(RoleName=DMS_VPC_ROLE_NAME)
    except aws.iam.exceptions.NoSuchEntityException:
        aws.iam.create_role(
            RoleName=DMS_VPC_ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
        )
        created_role = True
    aws.iam.attach_role_policy(RoleName=DMS_VPC_ROLE_NAME, PolicyArn=DMS_VPC_POLICY_ARN)
    print(f"✓ IAM role cho VPC: {DMS_VPC_ROLE_NAME}")

    try:
        role = aws.iam.get_role(RoleName=config.s3_role_name)["Role"]
    except aws.iam.exceptions.NoSuchEntityException:
        role = aws.iam.create_role(
            RoleName=config.s3_role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
        )["Role"]
        created_role = True

    s3_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:ListBucket"],
                "Resource": f"arn:aws:s3:::{config.s3_bucket}",
            },
            {
                "Effect": "Allow",
                "Action": [
                    "s3:PutObject",
                    "s3:PutObjectTagging",
                    "s3:PutObjectAcl",
                    "s3:GetObject",
                    "s3:DeleteObject",
                ],
                "Resource": f"arn:aws:s3:::{config.s3_bucket}/{config.s3_prefix}/*",
            },
        ],
    }
    aws.iam.put_role_policy(
        RoleName=config.s3_role_name,
        PolicyName="DmsS3Access",
        PolicyDocument=json.dumps(s3_policy),
    )
    print(f"✓ IAM role cho S3: {config.s3_role_name}")
    if created_role:
        print("  Chờ AWS đồng bộ IAM role...")
        time.sleep(10)
    return role["Arn"]


def _ensure_subnet_group(config: Config, aws: Aws) -> None:
    try:
        aws.dms.create_replication_subnet_group(
            ReplicationSubnetGroupIdentifier=config.subnet_group_id,
            ReplicationSubnetGroupDescription="DMS PostgreSQL RDS to S3",
            SubnetIds=config.subnet_ids,
        )
        print(f"✓ Đã tạo subnet group: {config.subnet_group_id}")
    except aws.dms.exceptions.ResourceAlreadyExistsFault:
        print(f"✓ Subnet group đã tồn tại: {config.subnet_group_id}")


def _route_table_id(vpc_id: str, subnet_id: str, aws: Aws) -> str:
    tables = aws.ec2.describe_route_tables(
        Filters=[{"Name": "association.subnet-id", "Values": [subnet_id]}]
    )["RouteTables"]
    if tables:
        return tables[0]["RouteTableId"]
    main_tables = aws.ec2.describe_route_tables(Filters=[
        {"Name": "vpc-id", "Values": [vpc_id]},
        {"Name": "association.main", "Values": ["true"]},
    ])["RouteTables"]
    if not main_tables:
        raise RuntimeError(f"Không tìm thấy route table cho subnet {subnet_id}")
    return main_tables[0]["RouteTableId"]


def _ensure_s3_gateway_endpoint(config: Config, aws: Aws) -> None:
    group = aws.dms.describe_replication_subnet_groups(
        Filters=[{"Name": "replication-subnet-group-id", "Values": [config.subnet_group_id]}]
    )["ReplicationSubnetGroups"][0]
    vpc_id = group["VpcId"]
    service_name = f"com.amazonaws.{config.aws_region}.s3"
    route_table_ids = sorted({_route_table_id(vpc_id, subnet_id, aws) for subnet_id in config.subnet_ids})
    endpoints = aws.ec2.describe_vpc_endpoints(Filters=[
        {"Name": "vpc-id", "Values": [vpc_id]},
        {"Name": "service-name", "Values": [service_name]},
        {"Name": "vpc-endpoint-type", "Values": ["Gateway"]},
    ])["VpcEndpoints"]

    if endpoints:
        endpoint = endpoints[0]
        missing = sorted(set(route_table_ids) - set(endpoint.get("RouteTableIds", [])))
        if missing:
            aws.ec2.modify_vpc_endpoint(VpcEndpointId=endpoint["VpcEndpointId"], AddRouteTableIds=missing)
        endpoint_id = endpoint["VpcEndpointId"]
    else:
        endpoint_id = aws.ec2.create_vpc_endpoint(
            VpcId=vpc_id,
            ServiceName=service_name,
            VpcEndpointType="Gateway",
            RouteTableIds=route_table_ids,
        )["VpcEndpoint"]["VpcEndpointId"]

    # Botocore cũ không có waiter cho VPC Endpoint, nên poll trực tiếp.
    deadline = time.monotonic() + 300
    while True:
        endpoint = aws.ec2.describe_vpc_endpoints(
            VpcEndpointIds=[endpoint_id]
        )["VpcEndpoints"][0]
        state = endpoint["State"]
        if state == "available":
            break
        if state in {"failed", "rejected"}:
            raise RuntimeError(
                f"S3 Gateway Endpoint {endpoint_id} thất bại: {state}"
            )
        if time.monotonic() >= deadline:
            raise TimeoutError(
                f"S3 Gateway Endpoint chưa available sau 300 giây; state={state}"
            )
        print(f"  Đang chờ S3 Gateway Endpoint: {state}")
        time.sleep(5)

    print(f"✓ Private route tới S3: {endpoint_id}")


def _ensure_instance(config: Config, aws: Aws) -> str:
    instance = _find_instance(config, aws)
    if instance is None:
        aws.dms.create_replication_instance(
            ReplicationInstanceIdentifier=config.instance_id,
            ReplicationInstanceClass=config.dms_instance_class,
            AllocatedStorage=config.dms_storage_gb,
            VpcSecurityGroupIds=config.security_group_ids,
            ReplicationSubnetGroupIdentifier=config.subnet_group_id,
            PubliclyAccessible=False,
        )
        print(f"✓ Đã yêu cầu tạo DMS instance: {config.instance_id}")
    else:
        print(f"✓ DMS instance đã tồn tại: {config.instance_id}")

    instance = _wait_for(
        "DMS instance available",
        lambda: _find_instance(config, aws),
        lambda item: item is not None and item["ReplicationInstanceStatus"] == "available",
    )
    return instance["ReplicationInstanceArn"]


In [3]:
from __future__ import annotations

def _ensure_endpoints(config: Config, aws: Aws, s3_role_arn: str) -> tuple[str, str]:
    source = _find_endpoint(config.source_endpoint_id, aws)
    if source is None:
        source = aws.dms.create_endpoint(
            EndpointIdentifier=config.source_endpoint_id,
            EndpointType="source",
            EngineName="postgres",
            ServerName=config.rds_host,
            Port=config.rds_port,
            DatabaseName=config.rds_database,
            Username=config.rds_username,
            Password=config.rds_password,
            SslMode="require",
        )["Endpoint"]
        print(f"✓ Đã tạo source endpoint: {config.source_endpoint_id}")
    else:
        print(f"✓ Source endpoint đã tồn tại: {config.source_endpoint_id}")

    target = _find_endpoint(config.target_endpoint_id, aws)
    if target is None:
        target = aws.dms.create_endpoint(
            EndpointIdentifier=config.target_endpoint_id,
            EndpointType="target",
            EngineName="s3",
            S3Settings={
                "BucketName": config.s3_bucket,
                "BucketFolder": config.s3_prefix,
                "ServiceAccessRoleArn": s3_role_arn,
                "DataFormat": "parquet",
                "CompressionType": "GZIP",
            },
        )["Endpoint"]
        print(f"✓ Đã tạo target endpoint: {config.target_endpoint_id}")
    else:
        print(f"✓ Target endpoint đã tồn tại: {config.target_endpoint_id}")
    return source["EndpointArn"], target["EndpointArn"]


def _test_endpoints(
    instance_arn: str,
    source_arn: str,
    target_arn: str,
    aws: Aws,
    timeout_seconds: int = 600,
) -> None:
    endpoint_arns = {"RDS": source_arn, "S3": target_arn}
    for name, endpoint_arn in endpoint_arns.items():
        try:
            aws.dms.test_connection(
                ReplicationInstanceArn=instance_arn,
                EndpointArn=endpoint_arn,
            )
            print(f"  Bắt đầu test kết nối {name}")
        except aws.dms.exceptions.InvalidResourceStateFault:
            print(f"  Test kết nối {name} đang chạy")

    time.sleep(5)  # DMS cần vài giây để cập nhật kết quả test mới.
    deadline = time.monotonic() + timeout_seconds
    while True:
        statuses: dict[str, str] = {}
        failures: dict[str, str] = {}
        for name, endpoint_arn in endpoint_arns.items():
            connections = aws.dms.describe_connections(
                Filters=[{"Name": "endpoint-arn", "Values": [endpoint_arn]}]
            )["Connections"]
            status = connections[0]["Status"] if connections else "not-tested"
            statuses[name] = status
            if status == "failed":
                failures[name] = connections[0].get("LastFailureMessage", "Không có chi tiết")

        if failures:
            details = "; ".join(f"{name}: {message}" for name, message in failures.items())
            raise RuntimeError(f"Test endpoint thất bại — {details}")
        if all(status == "successful" for status in statuses.values()):
            print("✓ Kết nối RDS và S3 đều thành công")
            return
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Test endpoint quá {timeout_seconds} giây: {statuses}")
        print(f"  Đang test endpoint: {statuses}")
        time.sleep(10)


def _table_mapping(config: Config) -> tuple[dict[str, Any], str]:
    # DMS hỗ trợ timestamp dạng YYYY-MM-DD HH:MM:SS.SSS. Giữ đầy đủ thời gian
    # để retention chính xác 90 * 24 giờ, thay vì vô tình làm tròn về 00:00 UTC.
    cutoff = (datetime.now(timezone.utc) - timedelta(days=config.retention_days)).strftime("%Y-%m-%d %H:%M:%S.000")
    mapping = {
        "rules": [{
            "rule-type": "selection",
            "rule-id": "1",
            "rule-name": "archive-old-rows",
            "object-locator": {
                "schema-name": config.source_schema,
                "table-name": config.source_table,
            },
            "rule-action": "include",
            # DMS OR các conditions trong cùng một filter, nhưng AND
            # các filter riêng. Vì vậy notnull và lte phải tách ra.
            "filters": [
                {
                    "filter-type": "source",
                    "column-name": config.date_column,
                    "filter-conditions": [{"filter-operator": "notnull"}],
                },
                {
                    "filter-type": "source",
                    "column-name": config.date_column,
                    "filter-conditions": [{"filter-operator": "lte", "value": cutoff}],
                },
            ],
        }],
    }
    return mapping, cutoff


def _effective_cutoff(task: dict[str, Any], config: Config) -> str:
    """Validate mapping thực tế trên AWS và trả về cutoff mà task đang dùng."""
    try:
        mapping = json.loads(task["TableMappings"])
        rules = mapping["rules"]
        selection_rules = [rule for rule in rules if rule.get("rule-type") == "selection"]
        if len(selection_rules) != 1:
            raise ValueError("cần đúng một selection rule")

        rule = selection_rules[0]
        locator = rule.get("object-locator", {})
        if rule.get("rule-action") != "include":
            raise ValueError("selection rule không phải include")
        if locator.get("schema-name") != config.source_schema or locator.get("table-name") != config.source_table:
            raise ValueError("selection rule trỏ sai source table")

        conditions_by_column: dict[str, list[dict[str, Any]]] = {}
        for source_filter in rule.get("filters", []):
            conditions_by_column.setdefault(source_filter.get("column-name", ""), []).extend(
                source_filter.get("filter-conditions", [])
            )
        date_conditions = conditions_by_column.get(config.date_column, [])
        has_notnull = any(item.get("filter-operator") == "notnull" for item in date_conditions)
        lte_values = [item.get("value") for item in date_conditions if item.get("filter-operator") == "lte"]
        if not has_notnull or len(lte_values) != 1 or not lte_values[0]:
            raise ValueError(f"thiếu filter {config.date_column} IS NOT NULL AND <= cutoff")
        return str(lte_values[0])
    except (KeyError, TypeError, json.JSONDecodeError, ValueError) as error:
        raise RuntimeError(
            f"Task {config.task_id} đang dùng table mapping không an toàn: {error}. "
            "Hãy chạy destroy(), dùng raw S3 prefix trống, rồi chạy setup() để tạo task mới."
        ) from error


def _ensure_and_start_task(
    config: Config,
    aws: Aws,
    instance_arn: str,
    source_arn: str,
    target_arn: str,
) -> tuple[str, str]:
    mapping, cutoff = _table_mapping(config)
    task = _find_task(config, aws)
    if task is None:
        task = aws.dms.create_replication_task(
            ReplicationTaskIdentifier=config.task_id,
            SourceEndpointArn=source_arn,
            TargetEndpointArn=target_arn,
            ReplicationInstanceArn=instance_arn,
            MigrationType="full-load",
            TableMappings=json.dumps(mapping),
            ReplicationTaskSettings=json.dumps({
                "FullLoadSettings": {
                    "TargetTablePrepMode": "DO_NOTHING",
                    "MaxFullLoadSubTasks": 8,
                    "CommitRate": 10_000,
                },
                "Logging": {"EnableLogging": True},
            }),
        )["ReplicationTask"]
        print(f"✓ Đã tạo replication task: {config.task_id}")
    elif task["Status"] == "ready":
        aws.dms.modify_replication_task(
            ReplicationTaskArn=task["ReplicationTaskArn"],
            TableMappings=json.dumps(mapping),
        )
        print(f"✓ Đã cập nhật table mapping: {config.task_id}")
    else:
        effective_cutoff = _effective_cutoff(task, config)
        print(f"✓ Replication task đã tồn tại: {config.task_id} ({task['Status']})")
        print(f"✓ Mapping thực tế trên AWS: {config.date_column} <= {effective_cutoff}")

    deadline = time.monotonic() + 300
    while True:
        task = _find_task(config, aws)
        if task is None:
            raise RuntimeError("Replication task biến mất trong lúc setup")
        task_status = task["Status"]
        if task_status == "ready":
            aws.dms.start_replication_task(
                ReplicationTaskArn=task["ReplicationTaskArn"],
                StartReplicationTaskType="start-replication",
            )
            print("✓ Đã bắt đầu full load")
            return task["ReplicationTaskArn"], cutoff
        if task_status in {"starting", "running"}:
            cutoff = _effective_cutoff(task, config)
            print(f"✓ Full load đang {task_status}")
            return task["ReplicationTaskArn"], cutoff
        if task_status == "stopped":
            errors = task.get("ReplicationTaskStats", {}).get("TablesErrored", 0)
            if errors == 0:
                cutoff = _effective_cutoff(task, config)
                print("✓ Full load trước đó đã hoàn tất; không chạy lại để tránh file trùng trên S3")
                return task["ReplicationTaskArn"], cutoff
            raise RuntimeError("Task đã dừng và có table lỗi. Xem status(); sau đó destroy() và setup() để chạy mới.")
        if task_status == "failed":
            raise RuntimeError(f"Task thất bại: {task.get('LastFailureMessage', 'Không có chi tiết')}")
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Task không chuyển sang ready; status={task_status}")
        print(f"  Đang chờ task sẵn sàng: {task_status}")
        time.sleep(10)


## 3. Ba lệnh chính

`setup()` có tính **idempotent**: gọi lại sẽ tái sử dụng resource đã có. Một full load đã hoàn tất sẽ không tự chạy lại, vì chạy lại có thể tạo file trùng trên S3.


In [4]:
from __future__ import annotations

def setup() -> None:
    """Tạo/tái sử dụng pipeline, test hai endpoint và bắt đầu full load."""
    config, aws = _context()
    print("=" * 72)
    print("SETUP — PostgreSQL RDS → S3")
    print("=" * 72)
    print(f"Nguồn : {config.source_schema}.{config.source_table}")
    print(f"Bộ lọc: {config.date_column} cũ hơn {config.retention_days} ngày")
    print(f"Đích  : s3://{config.s3_bucket}/{config.s3_prefix}/")

    _ensure_bucket(config, aws)
    s3_role_arn = _ensure_iam_roles(config, aws)
    _ensure_subnet_group(config, aws)
    _ensure_s3_gateway_endpoint(config, aws)
    instance_arn = _ensure_instance(config, aws)
    source_arn, target_arn = _ensure_endpoints(config, aws, s3_role_arn)
    _test_endpoints(instance_arn, source_arn, target_arn, aws)
    _, cutoff = _ensure_and_start_task(config, aws, instance_arn, source_arn, target_arn)

    print("-" * 72)
    print(f"Pipeline đã sẵn sàng. Điều kiện thực tế: {config.date_column} <= {cutoff}")
    print("Chạy status() để theo dõi tiến độ.")


def status() -> None:
    """In trạng thái ngắn gọn và đủ thông tin để tìm lỗi phổ biến."""
    config, aws = _context()
    print("=" * 72)
    print("STATUS")
    print("=" * 72)

    instance = _find_instance(config, aws)
    instance_status = instance["ReplicationInstanceStatus"] if instance else "not-found"
    print(f"DMS instance : {instance_status}")

    for label, endpoint_id in [
        ("RDS endpoint", config.source_endpoint_id),
        ("S3 endpoint ", config.target_endpoint_id),
    ]:
        endpoint = _find_endpoint(endpoint_id, aws)
        if endpoint is None:
            print(f"{label}: not-found")
            continue
        connections = aws.dms.describe_connections(
            Filters=[{"Name": "endpoint-arn", "Values": [endpoint["EndpointArn"]]}]
        )["Connections"]
        connection = connections[0] if connections else {}
        connection_status = connection.get("Status", "not-tested")
        print(f"{label}: {connection_status}")
        if connection_status == "failed":
            print(f"  Lỗi: {connection.get('LastFailureMessage', 'Không có chi tiết')}")

    task = _find_task(config, aws)
    if task is None:
        print("Task         : not-found — hãy chạy setup()")
    else:
        stats = task.get("ReplicationTaskStats", {})
        print(f"Task         : {task['Status']}")
        print(
            "Tiến độ      : "
            f"{stats.get('FullLoadProgressPercent', 0)}% | "
            f"completed={stats.get('TablesCompleted', 0)} | "
            f"loading={stats.get('TablesLoading', 0)} | "
            f"queued={stats.get('TablesQueued', 0)} | "
            f"errors={stats.get('TablesErrored', 0)}"
        )
        failure = task.get("LastFailureMessage")
        if failure:
            print(f"Task error   : {failure}")

        try:
            tables = aws.dms.describe_table_statistics(
                ReplicationTaskArn=task["ReplicationTaskArn"]
            )["TableStatistics"]
            for table in tables:
                print(
                    f"Table        : {table['SchemaName']}.{table['TableName']} | "
                    f"state={table['TableState']} | rows={table.get('FullLoadRows', 0)}"
                )
                if table.get("LastFailureMessage"):
                    print(f"  Lỗi table: {table['LastFailureMessage']}")
        except aws.dms.exceptions.InvalidResourceStateFault:
            pass  # Task mới tạo có thể chưa có table statistics.

    print(f"S3           : s3://{config.s3_bucket}/{config.s3_prefix}/")
    try:
        response = aws.s3.list_objects_v2(
            Bucket=config.s3_bucket,
            Prefix=f"{config.s3_prefix}/",
            MaxKeys=10,
        )
        objects = response.get("Contents", [])
        print(f"S3 objects   : {'chưa có file' if not objects else f'hiển thị {len(objects)} file đầu tiên'}")
        for item in objects:
            print(f"  - {item['Key']} ({item['Size']:,} bytes)")
    except ClientError as error:
        print(f"S3 error     : {error}")


In [5]:
from __future__ import annotations

def destroy() -> None:
    """Xóa tài nguyên DMS của pipeline; giữ nguyên RDS, bucket và dữ liệu S3."""
    config, aws = _context()
    print("=" * 72)
    print("DESTROY — chỉ xóa tài nguyên DMS")
    print("=" * 72)

    def find_or_none(fetch: Any) -> Any | None:
        """Coi ResourceNotFoundFault là resource đã được xóa thành công."""
        try:
            return fetch()
        except aws.dms.exceptions.ResourceNotFoundFault:
            return None

    find_task = lambda: find_or_none(lambda: _find_task(config, aws))
    find_endpoint = lambda endpoint_id: find_or_none(
        lambda: _find_endpoint(endpoint_id, aws)
    )
    find_instance = lambda: find_or_none(lambda: _find_instance(config, aws))

    task = find_task()
    if task is not None:
        active_statuses = {"running", "starting", "modifying", "stopping"}
        if task["Status"] in active_statuses:
            if task["Status"] != "stopping":
                try:
                    aws.dms.stop_replication_task(ReplicationTaskArn=task["ReplicationTaskArn"])
                except aws.dms.exceptions.InvalidResourceStateFault:
                    pass
            task = _wait_for(
                "task dừng",
                find_task,
                lambda item: item is None or item["Status"] not in active_statuses,
            )
        if task is not None and task["Status"] != "deleting":
            aws.dms.delete_replication_task(ReplicationTaskArn=task["ReplicationTaskArn"])
        _wait_for("task được xóa", find_task, lambda item: item is None)
        print(f"✓ Đã xóa task: {config.task_id}")
    else:
        print(f"- Task không tồn tại: {config.task_id}")

    for endpoint_id in [config.source_endpoint_id, config.target_endpoint_id]:
        endpoint = find_endpoint(endpoint_id)
        if endpoint is not None:
            if endpoint["Status"] != "deleting":
                aws.dms.delete_endpoint(EndpointArn=endpoint["EndpointArn"])
            _wait_for(
                f"endpoint {endpoint_id} được xóa",
                lambda endpoint_id=endpoint_id: find_endpoint(endpoint_id),
                lambda item: item is None,
                interval_seconds=5,
            )
            print(f"✓ Đã xóa endpoint: {endpoint_id}")
        else:
            print(f"- Endpoint không tồn tại: {endpoint_id}")

    instance = find_instance()
    if instance is not None:
        if instance["ReplicationInstanceStatus"] != "deleting":
            aws.dms.delete_replication_instance(
                ReplicationInstanceArn=instance["ReplicationInstanceArn"]
            )
        _wait_for("DMS instance được xóa", find_instance, lambda item: item is None)
        print(f"✓ Đã xóa instance: {config.instance_id}")
    else:
        print(f"- Instance không tồn tại: {config.instance_id}")

    try:
        aws.dms.delete_replication_subnet_group(
            ReplicationSubnetGroupIdentifier=config.subnet_group_id
        )
        print(f"✓ Đã xóa subnet group: {config.subnet_group_id}")
    except aws.dms.exceptions.ResourceNotFoundFault:
        print(f"- Subnet group không tồn tại: {config.subnet_group_id}")

    try:
        try:
            aws.iam.delete_role_policy(RoleName=config.s3_role_name, PolicyName="DmsS3Access")
        except aws.iam.exceptions.NoSuchEntityException:
            pass
        aws.iam.delete_role(RoleName=config.s3_role_name)
        print(f"✓ Đã xóa IAM role: {config.s3_role_name}")
    except aws.iam.exceptions.NoSuchEntityException:
        print(f"- IAM role không tồn tại: {config.s3_role_name}")

    print("-" * 72)
    print("Đã giữ nguyên: RDS, S3 bucket, dữ liệu S3, dms-vpc-role và S3 Gateway Endpoint.")


## 4. Chạy pipeline

Bỏ dấu `#` ở **một lệnh** cần chạy. Thông thường: chạy `setup()` một lần, gọi `status()` nhiều lần, và chỉ gọi `destroy()` khi đã kiểm tra dữ liệu S3 xong.


In [6]:
from __future__ import annotations

setup()


SETUP — PostgreSQL RDS → S3
Nguồn : public.orders
Bộ lọc: created_at_utc cũ hơn 90 ngày
Đích  : s3://my-data-lake-archival-demo/raw/rds/orders/
✓ Đã tạo S3 bucket: my-data-lake-archival-demo
✓ IAM role cho VPC: dms-vpc-role
✓ IAM role cho S3: orders-initial-s3-role
  Chờ AWS đồng bộ IAM role...
✓ Đã tạo subnet group: orders-initial-subnet
✓ Private route tới S3: vpce-0a8afdf2e593f39da
✓ Đã yêu cầu tạo DMS instance: orders-initial-instance
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  Đang chờ DMS instance available...
  

In [8]:
status()

STATUS
DMS instance : available
RDS endpoint: successful
S3 endpoint : successful
Task         : stopped
Tiến độ      : 100% | completed=0 | loading=0 | queued=0 | errors=0
Table        : public.orders | state=Table completed | rows=721
S3           : s3://my-data-lake-archival-demo/raw/rds/orders/
S3 objects   : hiển thị 1 file đầu tiên
  - raw/rds/orders/public/orders/LOAD00000001.parquet (12,178 bytes)


In [ ]:
destroy()

DESTROY — chỉ xóa tài nguyên DMS
  Đang chờ task được xóa...
  Đang chờ task được xóa...
✓ Đã xóa task: orders-initial-task
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ endpoint orders-initial-source được xóa...
  Đang chờ e